In [ ]:
import pandas as pd
import pymupdf


print("Pandas version:", pd.__version__)
print("PyMuPDF imported successfully!")

In [ ]:
from pathlib import Path

pdf_path=Path(r'E:\COLLEGEPUR\WORKS\AIML work\CBSE_AI_Assessment\data\raw\class9\maths\CBSE Class 9 Maths Sample Question Paper (Set 1) 2026-27 FREE PDF.pdf')

print('PDF exists:',pdf_path.exists())
print('PDF path:',pdf_path)

In [ ]:
doc=pymupdf.open(pdf_path)

print('PDF opened successfully')
print('Number_of_pages:',len(doc))

In [ ]:
page=doc[0]
text=page.get_text('text')
print(text)


In [ ]:
for page_number,page in enumerate(doc,start=1):
    text=page.get_text('text')
    print(f"Page{page}:{len(text)} characters")

In [ ]:
page=doc[2]

image_info=page.get_image_info(xrefs=True)

for i,image in enumerate(image_info, start=1):
    print(
        f"Image {i}: "
        f"width={image['width']}, "
        f"height={image['height']}, "
        f"bbox={image['bbox']}"
    )

In [ ]:
for page_number in [1, 3, 13]:
    page = doc[page_number - 1]

    print(f"\n--- PDF Page {page_number} ---")

    image_info = page.get_image_info(xrefs=True)

    for i, image in enumerate(image_info, start=1):
        print(
            f"Image {i}: "
            f"width={image['width']}, "
            f"height={image['height']}"
        )


In [ ]:
from pathlib import Path
raw_folder = Path(r"E:\COLLEGEPUR\WORKS\AIML work\CBSE_AI_Assessment\data\raw")
pdf_files = list(raw_folder.rglob("*.pdf"))
print("Total PDF files found:", len(pdf_files))

for pdf in pdf_files:
    print(pdf)

In [ ]:
all_pages=[]
for pdf_path in pdf_files:
    doc=pymupdf.open(pdf_path)

    for page_number,page in enumerate(doc,start=1):
        text=page.get_text("text").strip()
        all_pages.append({
            "source_file": pdf_path.name,
            "source_path": str(pdf_path),
            "page_number": page_number,
            "text": text,
            "text_length": len(text)
        })

all_pages

In [ ]:
import shutil
tesseract_path=shutil.which("tesseract")
print("Tesseract path:",tesseract_path)

In [ ]:
import pytesseract
print('pytesseract installed succesfully')

In [ ]:
import shutil
tesseract_path=shutil.which("tesseract")
print("Tesseract path:",tesseract_path)

In [ ]:
import pytesseract
pytesseract.pytesseract.tesseract_cmd= r"C:\Program Files\Tesseract-OCR\tesseract.exe"
print(pytesseract.get_tesseract_version())

In [ ]:
for i, pdf in enumerate(pdf_files[:10]):
    print(i, pdf)

In [ ]:
class9_pdfs = [
    pdf for pdf in pdf_files
    if "class9" in [part.lower() for part in pdf.parts]
]

print("Class 9 PDFs:", len(class9_pdfs))

for pdf in class9_pdfs:
    print(pdf)

In [ ]:
from PIL import Image
import io
pdf_path = class9_pdfs[0]
doc = pymupdf.open(pdf_path)
page = doc[0]
pix = page.get_pixmap(dpi=200)

image = Image.open(io.BytesIO(pix.tobytes("png")))
ocr_text = pytesseract.image_to_string(image)
print(ocr_text[:2000])
doc.close()

In [ ]:
all_extracted_pages=[]

for pdf_path in pdf_files:
      # Find class from folder name
    class_name = "class9" if "class9" in [part.lower() for part in pdf_path.parts] else "class10"

    # Find subject from folder name
    subject = pdf_path.parent.name

    print(f"Processing: {class_name} | {subject} | {pdf_path.name}")

    doc = pymupdf.open(pdf_path)

    for page_number, page in enumerate(doc, start=1):

        # First try normal PDF text extraction
        native_text = page.get_text("text").strip()

        # OCR only when native text is very little
        ocr_text = ""

        if len(native_text) < 150:

            pix = page.get_pixmap(dpi=200)

            image = Image.open(
                io.BytesIO(pix.tobytes("png"))
            )

            ocr_text = pytesseract.image_to_string(image).strip()

        # Choose the better available text
        final_text = native_text if len(native_text) >= 150 else ocr_text

        all_extracted_pages.append({
            "class": class_name,
            "subject": subject,
            "source_file": pdf_path.name,
            "page_number": page_number,
            "native_text": native_text,
            "ocr_text": ocr_text,
            "final_text": final_text
        })

    doc.close()

print("\nTotal pages processed:", len(all_extracted_pages))    


In [ ]:
df_pages = pd.DataFrame(all_extracted_pages)

print("Total page rows:", len(df_pages))
print("Total PDFs:", df_pages["source_file"].nunique())

print("\nClass-wise pages:")
print(df_pages["class"].value_counts())

print("\nSubject-wise pages:")
print(df_pages["subject"].value_counts())

In [ ]:
print("Pages that used OCR:", (df_pages["ocr_text"] != "").sum())
print("Pages with no final text:", (df_pages["final_text"].str.strip() == "").sum())

In [ ]:
import re
import pandas as pd

question_rows = []

for (class_name, subject, source_file), group in df_pages.groupby(
    ["class", "subject", "source_file"]
):
    
    group = group.sort_values("page_number")
    full_text = "\n".join(group["final_text"].fillna(""))

    # Remove text before the first section
    section_match = re.search(
        r"\bSECTION\s*[-–]?\s*[A-E]\b",
        full_text,
        flags=re.IGNORECASE
    )

    if section_match:
        full_text = full_text[section_match.start():]

    # Find numbered questions: 1. / 1) / 2. / 2)
    pattern = r"(?m)^\s*(\d{1,3})[\.\)]\s+(.*?)(?=^\s*\d{1,3}[\.\)]\s+|\Z)"

    matches = re.finditer(
        pattern,
        full_text,
        flags=re.MULTILINE | re.DOTALL
    )

    for match in matches:
        question_number = int(match.group(1))
        question_text = re.sub(r"\s+", " ", match.group(2)).strip()

        # Ignore very short accidental matches
        if len(question_text) < 20:
            continue

        session_match = re.search(r"(20\d{2}-\d{2})", source_file)
        session = session_match.group(1) if session_match else "Unknown"

        question_rows.append({
            "question_id": len(question_rows) + 1,
            "class": class_name,
            "subject": subject,
            "session": session,
            "source_file": source_file,
            "question_number": question_number,
            "question_text": question_text,
            "answer": ""
        })

question_df = pd.DataFrame(question_rows)

print("Total questions extracted:", len(question_df))
display(question_df.head(10))

In [ ]:
from pathlib import Path

processed_folder = raw_folder.parent / "processed"
processed_folder.mkdir(exist_ok=True)

print("Processed folder:", processed_folder)

In [ ]:
output_file = processed_folder / "question_dataset.csv"

question_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset saved:")
print(output_file)

In [ ]:
# Create a richer question-bank dataset from the extracted questions

question_bank = question_df.copy()

# Section
question_bank["section"] = ""

# Question type
question_bank["question_type"] = ""

# Options
question_bank["options"] = ""

# Marks
question_bank["marks"] = ""

# Topic - will be populated during subject-wise cleaning
question_bank["topic"] = ""

# Difficulty - will be assigned later
question_bank["difficulty"] = ""

# Diagram/image indicator
question_bank["has_diagram"] = False

# Answer is unavailable in the current raw question set
question_bank["correct_answer"] = ""

# Explanation is unavailable in the current raw question set
question_bank["answer_explanation"] = ""

print("Rows:", len(question_bank))
print("Columns:")
print(question_bank.columns.tolist())

display(question_bank.head(10))

In [ ]:
import re

def detect_question_type(text):
    text_lower = text.lower()

    if re.search(r"\(a\).*?\(b\).*?\(c\).*?\(d\)", text, re.DOTALL):
        return "MCQ"
    elif "assertion" in text_lower and "reason" in text_lower:
        return "Assertion-Reason"
    elif "case-based" in text_lower or "passage" in text_lower:
        return "Case-Based"
    else:
        return "Short/Long Answer"


def extract_options(text):
    matches = re.findall(
        r"\([a-dA-D]\)\s*(.*?)(?=\s*\([a-dA-D]\)|$)",
        text,
        re.DOTALL
    )

    return " | ".join(x.strip() for x in matches)


question_bank["question_type"] = question_bank["question_text"].apply(
    detect_question_type
)

question_bank["options"] = question_bank["question_text"].apply(
    extract_options
)

display(
    question_bank[
        ["question_id", "class", "subject",
         "question_number", "question_type", "options",
         "question_text"]
    ].head(10)
)

In [ ]:
final_file = processed_folder / "question_bank_enriched.csv"

question_bank.to_csv(
    final_file,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(final_file)

In [ ]:
df=pd.read_csv('E:\COLLEGEPUR\WORKS\AIML work\CBSE_AI_Assessment\data\processed\question_bank_enriched.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.columns 

In [ ]:
display(question_bank.head(10).T)

In [ ]:
clean_bank = question_bank[
    ~question_bank["question_text"].str.startswith("Section", na=False)
].copy()

print("Before cleaning:", len(question_bank))
print("After removing section text:", len(clean_bank))

display(clean_bank.head(10).T)

In [ ]:
print("Empty questions:", clean_bank["question_text"].isna().sum())

print(
    "Very short questions:",
    (clean_bank["question_text"].fillna("").str.len() < 20).sum()
)

print(
    "Duplicate questions:",
    clean_bank.duplicated(
        subset=["class", "subject", "question_text"]
    ).sum()
)

In [ ]:
duplicates = clean_bank[
    clean_bank.duplicated(
        subset=["class", "subject", "question_text"],
        keep=False
    )
]

display(
    duplicates[
        ["class", "subject", "source_file", "question_text"]
    ].sort_values(["class", "subject", "question_text"]).head(20)
)

In [ ]:
instruction_patterns = (
    "all questions",
    "there is no overall choice",
    "draw neat figures",
    "general instructions",
    "(a) both",
    "(a) is correct",
    "(a) pearl explanation"
)

clean_bank = clean_bank[
    ~clean_bank["question_text"]
    .fillna("")
    .str.lower()
    .str.startswith(instruction_patterns)
].copy()

print("Rows after removing obvious non-questions:", len(clean_bank))

In [ ]:
print(
    "Duplicate questions:",
    clean_bank.duplicated(
        subset=["class", "subject", "question_text"]
    ).sum()
)

In [ ]:
duplicates = clean_bank[
    clean_bank.duplicated(
        subset=["class", "subject", "question_text"],
        keep=False
    )
]

display(
    duplicates[
        ["class", "subject", "source_file", "question_text"]
    ].head(10)
)

In [ ]:
duplicate_check = (
    duplicates
    .groupby(["source_file", "question_text"])
    .size()
    .reset_index(name="count")
)

display(
    duplicate_check[
        duplicate_check["count"] > 1
    ].head(10)
)

In [ ]:
clean_bank = clean_bank.drop_duplicates(
    subset=["class", "subject", "source_file", "question_text"],
    keep="first"
).copy()

print("Rows after removing exact duplicates:", len(clean_bank))

In [ ]:
print(clean_bank["session"].value_counts())

In [ ]:
import re

def get_session(filename):
    match = re.search(r"(20\d{2}-\d{2})", filename)
    
    if match:
        return match.group(1)
    
    match = re.search(r"(\d{2})-(\d{2})", filename)
    
    if match:
        return "20" + match.group(1) + "-" + match.group(2)
    
    return "Unknown"

clean_bank["session"] = clean_bank["source_file"].apply(get_session)

print(clean_bank["session"].value_counts())

In [ ]:
unknown_sessions = clean_bank[
    clean_bank["session"] == "Unknown"
]

display(
    unknown_sessions[
        ["class", "subject", "source_file"]
    ].drop_duplicates().head(30)
)

In [ ]:
print(
    unknown_sessions["source_file"]
    .drop_duplicates()
    .to_string(index=False)
)

In [ ]:
for filename in unknown_sessions["source_file"].drop_duplicates():

    pdf_path = next(pdf for pdf in pdf_files if pdf.name == filename)

    doc = pymupdf.open(pdf_path)

    first_page_text = doc[0].get_text("text")

    print("\n" + "=" * 80)
    print(filename)
    print("-" * 80)
    print(first_page_text[:1500])

    doc.close()

In [ ]:
clean_bank.loc[
    (clean_bank["class"] == "class9") &
    (clean_bank["session"] == "Unknown"),
    "session"
] = "2026-27"

print(clean_bank["session"].value_counts())

In [ ]:
print(clean_bank["question_type"].value_counts())

In [ ]:
display(
    clean_bank[
        clean_bank["question_type"] == "Short/Long Answer"
    ][
        ["class", "subject", "source_file", "question_number", "question_text"]
    ].head(15)
)

In [ ]:
mcq_pattern = r"(?is)(?:\(?a\)?[\.\):])[\s\S]+(?:\(?b\)?[\.\):])[\s\S]+(?:\(?c\)?[\.\):])[\s\S]+(?:\(?d\)?[\.\):])"

is_mcq = clean_bank["question_text"].fillna("").str.contains(
    mcq_pattern,
    regex=True
)

clean_bank.loc[is_mcq, "question_type"] = "MCQ"

print(clean_bank["question_type"].value_counts())

In [ ]:
# Restore the original question types
original_types = question_bank.set_index("question_id")["question_type"]

clean_bank["question_type"] = clean_bank["question_id"].map(original_types)

# Change only Short/Long Answer rows that clearly contain a,b,c,d options
is_mcq = clean_bank["question_text"].fillna("").str.contains(
    mcq_pattern,
    regex=True
)

clean_bank.loc[
    is_mcq & clean_bank["question_type"].eq("Short/Long Answer"),
    "question_type"
] = "MCQ"

print(clean_bank["question_type"].value_counts())

In [ ]:
display(
    clean_bank[
        clean_bank["question_type"] == "MCQ"
    ][
        ["class", "subject", "question_text"]
    ].head(10)
)

In [ ]:
# Fix Assertion-Reason questions first
is_assertion_reason = (
    clean_bank["question_text"]
    .fillna("")
    .str.contains("assertion", case=False)
    &
    clean_bank["question_text"]
    .fillna("")
    .str.contains("reason", case=False)
)

clean_bank.loc[is_assertion_reason, "question_type"] = "Assertion-Reason"

print(clean_bank["question_type"].value_counts())

In [ ]:
display(
    clean_bank[
        clean_bank["question_type"] == "Case-Based"
    ][
        ["class", "subject", "question_text"]
    ].head(10)
)

In [ ]:
display(
    clean_bank[
        clean_bank["question_type"] == "Case-Based"
    ][
        ["class", "subject", "source_file",
         "question_number", "question_text"]
    ]
)

In [ ]:
# Reset current Case-Based labels
clean_bank.loc[
    clean_bank["question_type"] == "Case-Based",
    "question_type"
] = "Short/Long Answer"

# Identify clearly case/passage-based questions
text = clean_bank["question_text"].fillna("")

is_case = (
    text.str.contains("case study", case=False, na=False)
    | text.str.contains("read the passage", case=False, na=False)
    | text.str.contains("read the following text", case=False, na=False)
)

clean_bank.loc[is_case, "question_type"] = "Case-Based"

print(clean_bank["question_type"].value_counts())

In [ ]:
print(
    "MCQs with empty options:",
    clean_bank.loc[
        clean_bank["question_type"] == "MCQ",
        "options"
    ].fillna("").str.strip().eq("").sum()
)

In [ ]:
display(
    clean_bank[
        (clean_bank["question_type"] == "MCQ") &
        (clean_bank["options"].fillna("").str.strip() == "")
    ][
        ["class", "subject", "question_text"]
    ].head(5)
)

In [ ]:
for _, row in clean_bank[
    (clean_bank["question_type"] == "MCQ") &
    (clean_bank["options"].fillna("").str.strip() == "")
].head(5).iterrows():

    print("\nQUESTION:")
    print(row["question_text"])
    print("-" * 80)

In [ ]:
def extract_options(text):
    pattern = r"(?i)(?:^|\s)([a-d])\.\s*(.*?)(?=\s+[a-d]\.\s+|$)"
    matches = re.findall(pattern, text)

    return " | ".join(
        f"{letter}. {option.strip()}"
        for letter, option in matches
    )

clean_bank["options"] = clean_bank["question_text"].apply(extract_options)

print(
    "MCQs with options:",
    clean_bank.loc[
        clean_bank["question_type"] == "MCQ",
        "options"
    ].ne("").sum()
)

In [ ]:
display(
    clean_bank[
        (clean_bank["question_type"] == "MCQ") &
        (clean_bank["options"].fillna("").str.strip() == "")
    ][
        ["class", "subject", "source_file", "question_text"]
    ].head(5)
)

In [ ]:
for _, row in clean_bank[
    (clean_bank["question_type"] == "MCQ") &
    (clean_bank["options"].fillna("").str.strip() == "")
].head(5).iterrows():

    print("\nQUESTION:")
    print(row["question_text"])
    print("-" * 80)

In [ ]:
def extract_options(text):
    pattern = r"(?i)(?:^|\s)(\([a-d]\)|[a-d][\.\:\)])\s+(.+?)(?=\s+(?:\([a-d]\)|[a-d][\.\:\)])\s+|$)"
    
    matches = re.findall(pattern, text)

    return " | ".join(
        f"{label.upper()} {option.strip()}"
        for label, option in matches
    )


clean_bank["options"] = clean_bank["question_text"].apply(extract_options)

print(
    "MCQs with options:",
    clean_bank.loc[
        clean_bank["question_type"] == "MCQ",
        "options"
    ].ne("").sum()
)

In [ ]:
display(
    clean_bank[
        clean_bank["question_type"] == "MCQ"
    ][["question_text", "options"]].head(5)
)

In [ ]:
display(
    clean_bank[
        (clean_bank["question_type"] == "MCQ") &
        (clean_bank["options"].fillna("").str.strip() == "")
    ][
        ["class", "subject", "source_file", "question_text"]
    ]
)

In [ ]:
for _, row in clean_bank[
    (clean_bank["question_type"] == "MCQ") &
    (clean_bank["options"].fillna("").str.strip() == "")
].iterrows():

    print("\nQUESTION:")
    print(row["question_text"])
    print("-" * 80)

In [ ]:
def remove_options(text):
    return re.split(
        r"(?i)\s+(?:\([a-d]\)|[a-d][\.\:\)])\s+",
        text,
        maxsplit=1
    )[0].strip()

clean_bank["question_text_clean"] = (
    clean_bank["question_text"]
    .apply(remove_options)
)

display(
    clean_bank[
        ["question_text", "question_text_clean", "options"]
    ].head(5)
)

In [ ]:
clean_file = processed_folder / "clean_question_bank.csv"

clean_bank.to_csv(
    clean_file,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", clean_file)
print("Rows:", len(clean_bank))
print("Columns:", len(clean_bank.columns))

In [ ]:
print("Missing question text:", clean_bank["question_text_clean"].isna().sum())
print("Missing class:", clean_bank["class"].isna().sum())
print("Missing subject:", clean_bank["subject"].isna().sum())
print("Missing session:", clean_bank["session"].isna().sum())

In [ ]:
llm_dataset = clean_bank[
    [
        "question_id",
        "class",
        "subject",
        "session",
        "question_type",
        "question_text_clean",
        "options",
        "source_file"
    ]
].copy()

print("LLM dataset rows:", len(llm_dataset))
print("LLM dataset columns:", llm_dataset.columns.tolist())

display(llm_dataset.head(10))

In [ ]:
llm_file = processed_folder / "llm_input_dataset.csv"

llm_dataset.to_csv(
    llm_file,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", llm_file)

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / (1024 ** 3)

print(f"System RAM: {ram_gb:.1f} GB")

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

In [ ]:
from transformers import pipeline
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct"
)
print("Model loaded successfully!")

In [ ]:
messages = [
    {"role": "user",
     "content": "Write one simple Class 9 Maths question about linear equations."
    }]
result = generator(messages, max_new_tokens=100)
print(result)

In [ ]:
sample = llm_dataset[
    llm_dataset["question_type"] == "MCQ"
].iloc[0]

print("Class:", sample["class"])
print("Subject:", sample["subject"])
print("Type:", sample["question_type"])
print("Question:", sample["question_text_clean"])
print("Options:", sample["options"])

In [ ]:
sample = llm_dataset[
    llm_dataset["question_type"] == "MCQ"
].iloc[0]

print("Class:", sample["class"])
print("Subject:", sample["subject"])
print("Type:", sample["question_type"])
print("Question:", sample["question_text_clean"])
print("Options:", sample["options"])

In [ ]:
from transformers import GenerationConfig

generation_config = GenerationConfig(
    max_new_tokens=150,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.1
)

def generate_cbse_question(class_name, subject, question_type, topic):
    
    prompt = f"""
You are a CBSE question paper writer.

Generate ONE new {question_type} for:
Class: {class_name}
Subject: {subject}
Topic: {topic}

Rules:
- Do not copy existing questions.
- Keep the question suitable for the given class.
- For MCQ, give exactly 4 options: A, B, C, D.
- Give one correct answer.
- Do not give an explanation.
- Follow exactly this format:

QUESTION:
<question>

A) <option>
B) <option>
C) <option>
D) <option>

ANSWER:
<correct answer>
"""

    result = generator(
        [{"role": "user", "content": prompt}],
        generation_config=generation_config
    )

    return result[0]["generated_text"][-1]["content"]

In [ ]:
generated = generate_cbse_question(
    "class10",
    "computer_application",
    "MCQ",
    "Internet and web concepts"
)

print(generated)

In [ ]:
generated_file = processed_folder / "generated_questions.csv"
generated_df.to_csv(
    generated_file,
    index=False,
    encoding="utf-8-sig"
)
print("Saved:", generated_file)

In [ ]:
generated_file = processed_folder / "generated_questions.csv"

generated_df = pd.read_csv(generated_file)

print("Generated questions:", len(generated_df))
display(generated_df.head(5))

In [ ]:
display(
    generated_df[
        ["class", "subject", "source_question", "generated_question"]
    ].head(5)
)

In [ ]:
for i, row in generated_df.head(5).iterrows():
    print("\n" + "=" * 80)
    print("CLASS:", row["class"])
    print("SUBJECT:", row["subject"])
    print("SOURCE:", row["source_question"])
    print("\nGENERATED:")
    print(row["generated_question"])

In [ ]:
print("Generated:", len(generated_df))

for i, row in generated_df.iterrows():
    print(
        i,
        "|",
        row["class"],
        "|",
        row["subject"]
    )

In [ ]:
import re
from difflib import SequenceMatcher

def validate_generated(row):
    text = row["generated_question"]

    option_count = len(
        re.findall(r"(?m)^[A-D][\)\.]\s+", text)
    )

    has_answer = bool(
        re.search(r"(?im)^ANSWER:\s*$", text)
    )

    source = row["source_question"]
    similarity = SequenceMatcher(
        None,
        source.lower(),
        text.lower()
    ).ratio()

    if option_count == 4 and has_answer and similarity < 0.75:
        return "PASS_FORMAT"

    return "REVIEW"

generated_df["quality_status"] = generated_df.apply(
    validate_generated,
    axis=1
)

print(generated_df["quality_status"].value_counts())

display(
    generated_df[
        ["class", "subject", "quality_status"]
    ]
)

In [ ]:
for i, row in generated_df.iterrows():
    text = row["generated_question"]

    option_count = len(
        re.findall(r"(?m)^[A-D][\)\.]\s+", text)
    )

    has_answer = bool(
        re.search(r"(?im)^ANSWER:\s*$", text)
    )

    similarity = SequenceMatcher(
        None,
        row["source_question"].lower(),
        text.lower()
    ).ratio()

    print(
        i,
        "| options:", option_count,
        "| answer:", has_answer,
        "| similarity:", round(similarity, 2)
    )

In [ ]:
def validate_generated(row):
    text = row["generated_question"]

    # Count options only before the ANSWER section
    question_part = text.split("ANSWER:", 1)[0]

    option_count = len(
        re.findall(r"(?m)^[A-D][\)\.]\s+", question_part)
    )

    has_answer = "ANSWER:" in text

    similarity = SequenceMatcher(
        None,
        row["source_question"].lower(),
        text.lower()
    ).ratio()

    if option_count == 4 and has_answer and similarity < 0.75:
        return "PASS_FORMAT"

    return "REVIEW"


generated_df["quality_status"] = generated_df.apply(
    validate_generated,
    axis=1
)

print(generated_df["quality_status"].value_counts())

In [ ]:
display(
    generated_df[
        generated_df["quality_status"] == "REVIEW"
    ][
        ["class", "subject", "source_question", "generated_question"]
    ]
)

In [ ]:
replacement = generate_cbse_question(
    "class9",
    "social_science",
    "MCQ",
    "French Revolution and its major events"
)

print(replacement)

In [ ]:
accepted_df = generated_df[
    generated_df["quality_status"] == "PASS_FORMAT"
].copy()

replacement_row = {
    "class": "class9",
    "subject": "social_science",
    "question_type": "MCQ",
    "source_question": "French Revolution topic",
    "generated_question": replacement,
    "quality_status": "PASS_FORMAT",
    "answer_status": "Needs verification"
}

accepted_df["answer_status"] = "Needs verification"

accepted_df = pd.concat(
    [accepted_df, pd.DataFrame([replacement_row])],
    ignore_index=True
)

print("Final generated questions:", len(accepted_df))

In [ ]:
final_generated_file = processed_folder / "final_generated_questions.csv"

accepted_df.to_csv(
    final_generated_file,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", final_generated_file)